In [5]:
%pip install opencv-python numpy matplotlib scipy

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   - -------------------------------------- 1.3/44.0 MB 18.1 MB/s eta 0:00:03
   ------- -------------------------------- 8.4/44.0 MB 29.4 MB/s eta 0:00:02
   -------------- ------------------------- 16.3/44.0 MB 32.7 MB/s eta 0:00:01
   -------------------- ------------------- 22.8/44.0 MB 33.3 MB/s eta 0:00:01
   -------------------------- ------------- 29.4/44.0 MB 31.6 MB/s eta 0:00:01
   ---------------------------------- ----- 37.5/44.0 MB 32.7 MB/s eta 0:00:01
   ---------------------------------------  43.3/44.0 MB 31.7 MB/s eta 0:00:01
   ---------------------------------------- 44.0/44.0 MB 29.9 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\jonah\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import signal

In [7]:
def show_fit(window_name, img, max_dim=900):
    """Display img in a window, downscaling (never upscaling) so it fits
    within max_dim pixels on its longer side."""
    h, w = img.shape[:2]
    scale = min(max_dim / h, max_dim / w, 1.0)
    if scale < 1.0:
        img = cv2.resize(img, (int(w * scale), int(h * scale)))
    cv2.imshow(window_name, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


def normalize_for_display(im):
    """Map a signed float array (e.g. a derivative filter response) to
    uint8 [0, 255] via absolute value, scaled so the strongest edge is 255."""
    im = np.abs(im)
    m = im.max()
    if m > 0:
        im = im / m * 255.0
    return im.astype(np.uint8)

## Part 1.1: Convolution from scratch

In [8]:
def convolve_4loops(im, kernel):
    """Zero-padded 'same' convolution (kernel flipped) via four nested
    Python loops. Correct but slow -- O(H*W*kh*kw) at Python speed."""
    img_h, img_w = im.shape
    ker_h, ker_w = kernel.shape

    kernel = np.flipud(np.fliplr(kernel))
    pad_h = (ker_h - 1) // 2
    pad_w = (ker_w - 1) // 2
    padded_im = np.pad(im, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant', constant_values=0)

    output = np.zeros((img_h, img_w))
    for x in range(img_h):
        for y in range(img_w):
            for i in range(ker_h):
                for j in range(ker_w):
                    output[x, y] += padded_im[x + i, y + j] * kernel[i, j]

    return output


def convolve_2loops(im, kernel):
    """Zero-padded 'same' convolution (kernel flipped), vectorized over the
    kernel window so only the two spatial loops run in Python."""
    img_h, img_w = im.shape
    ker_h, ker_w = kernel.shape

    kernel = np.flipud(np.fliplr(kernel))
    pad_h = (ker_h - 1) // 2
    pad_w = (ker_w - 1) // 2
    padded_im = np.pad(im, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant', constant_values=0)

    output = np.zeros((img_h, img_w))
    for x in range(img_h):
        for y in range(img_w):
            region = padded_im[x:x + ker_h, y:y + ker_w]
            output[x, y] = np.sum(region * kernel)

    return output

## Part 1.1: Filter my own picture

In [9]:
im_selfie = cv2.imread('data/selfie.jpg', cv2.IMREAD_GRAYSCALE).astype(np.float64) / 255.0
box_filter = np.ones((9, 9)) / 81.0

im_4loops = convolve_4loops(im_selfie, box_filter)
im_box = convolve_2loops(im_selfie, box_filter)
im_scipy = signal.convolve2d(im_selfie, box_filter, mode='same', boundary='fill', fillvalue=0)
print('Do all results match?', np.allclose(im_4loops, im_box) and np.allclose(im_box, im_scipy))

dx_kernel = np.array([-1, 0, 1], dtype=np.float64).reshape(1, 3)
dy_kernel = np.array([-1, 0, 1], dtype=np.float64).reshape(3, 1)
im_dx = convolve_2loops(im_selfie, dx_kernel)
im_dy = convolve_2loops(im_selfie, dy_kernel)

show_fit('Original Image', im_selfie)
show_fit('Box filter', im_box)
show_fit('DX', normalize_for_display(im_dx))
show_fit('DY', normalize_for_display(im_dy))

Do all results match? True


## Part 1.2: Cameraman edges

In [10]:
im_cameraman = cv2.imread('data/cameraman.png', cv2.IMREAD_GRAYSCALE).astype(np.float64) / 255.0
im_cameraman_dx = signal.convolve2d(im_cameraman, dx_kernel, mode='same', boundary='fill', fillvalue=0)
im_cameraman_dy = signal.convolve2d(im_cameraman, dy_kernel, mode='same', boundary='fill', fillvalue=0)
im_gradient_magnitude = np.sqrt(im_cameraman_dx**2 + im_cameraman_dy**2)

edge_threshold = 50 / 255
im_cameraman_binarized = np.where(np.abs(im_gradient_magnitude) > edge_threshold, 255, 0).astype(np.uint8)

show_fit('Cameraman', im_cameraman)
show_fit('Cameraman DX', normalize_for_display(im_cameraman_dx))
show_fit('Cameraman DY', normalize_for_display(im_cameraman_dy))
show_fit('Cameraman Gradient Magnitude', normalize_for_display(im_gradient_magnitude))
show_fit('Cameraman Binarized', im_cameraman_binarized)

## Part 1.3: Cameraman edges (blurred first)

In [11]:
im_cameraman = cv2.imread('data/cameraman.png', cv2.IMREAD_GRAYSCALE).astype(np.float64) / 255.0
dx_kernel = np.array([-1, 0, 1], dtype=np.float64).reshape(1, 3)
dy_kernel = np.array([-1, 0, 1], dtype=np.float64).reshape(3, 1)
guassian_filter = cv2.getGaussianKernel(ksize=9, sigma=1.5) * cv2.getGaussianKernel(ksize=9, sigma=1.5).T
im_cameraman_blurred = signal.convolve2d(im_cameraman, guassian_filter, mode='same', boundary='fill', fillvalue=0)
im_cameraman_blurred_dx = signal.convolve2d(im_cameraman_blurred, dx_kernel, mode='same', boundary='fill', fillvalue=0)
im_cameraman_blurred_dy = signal.convolve2d(im_cameraman_blurred, dy_kernel, mode='same', boundary='fill', fillvalue=0)
im_gradient_magnitude = np.sqrt(im_cameraman_blurred_dx**2 + im_cameraman_blurred_dy**2)
edge_threshold = 25 / 255
im_cameraman_binarized = np.where(np.abs(im_gradient_magnitude) > edge_threshold, 255, 0).astype(np.uint8)

show_fit('Cameraman', im_cameraman)
show_fit('Cameraman Blurred', im_cameraman_blurred)
show_fit('Cameraman Blurred DX', normalize_for_display(im_cameraman_blurred_dx))
show_fit('Cameraman Blurred DY', normalize_for_display(im_cameraman_blurred_dy))
show_fit('Cameraman Blurred Gradient Magnitude', normalize_for_display(im_gradient_magnitude))
show_fit('Cameraman Blurred Binarized', im_cameraman_binarized)